In [ ]:
import numpy as np
import os
import scipy
from load_data_function import load_data,save_data
import re
from load_data_function import fig_plot,battery_soh_plot
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from load_data_function import smooth_soh

In [ ]:
def month_offset(m):
    """将实际月份转换为从8月开始的排序序号（8月=0, 9月=1...7月=11）"""
    m = int(m)
    return (m - 8) % 12  # 用模运算实现环形偏移


CALCE_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data\CALCE-dataset'
package_list=['package_1']
CALCE_data={}
CALCE_SOH={}
for i,package in enumerate(package_list):

    print(f'package: {package}')
    battery_list=os.listdir(CALCE_path)

    print(battery_list)

    package1={}
    package2={}
    for j,battery in enumerate(battery_list):
        print(f'battery: {battery}')
        battery_path=os.path.join(CALCE_path,battery)
        cycle_list = sorted(os.listdir(battery_path),
                    key=lambda x: (month_offset(x.split('_')[2]),
                                   int(x.split('_')[3])))

        #print(cycle_list)
        package1[f'battery_{j+1}']=[]
        package2[f'battery_{j+1}']=[]
        for cycle in cycle_list:
            print(cycle)
            cycle_path=os.path.join(battery_path,cycle)
            battery_data = pd.read_excel(cycle_path,sheet_name=1).dropna(subset=['Discharge_Capacity(Ah)', 'Charge_Capacity(Ah)'])
        #print(battery_data.shape)

            voltage=[]
            current=[]
            time=[]
            capacity=[]

            cycle_num= battery_data['Cycle_Index'].unique()
            cycle_num=sorted(cycle_num)
            #print(cycle_num)
            for k in cycle_num:
                cycle_data=battery_data[battery_data['Cycle_Index']==k]
                voltage=cycle_data['Voltage(V)'].values.reshape(1,-1)

                current=cycle_data['Current(A)'].values.reshape(1,-1)
                time_segment = cycle_data['Test_Time(s)'].values.reshape(1,-1)
                time=time_segment  # 转换为秒
                #time=time.reshape(1,-1)
                discharge_capacity=cycle_data['Discharge_Capacity(Ah)'].values.reshape(1,-1)
                if k==1:
                    discharge_capacity=discharge_capacity
                else:
                    discharge_capacity=discharge_capacity-discharge_capacity[0][0]
                    #print(discharge_capacity)
                charge_capacity=cycle_data['Charge_Capacity(Ah)'].values.reshape(1,-1)
                discharge_capacity=np.float32(discharge_capacity)
                #charge_capacity=np.float32(charge_capacity)
                #capacity=np.concatenate((discharge_capacity,charge_capacity),axis=1)
                if discharge_capacity.shape[1]<100:
                    continue
                #print(discharge_capacity.shape)
                capacity_max=np.max(discharge_capacity)

                soh=capacity_max/1.1

                #print(soh)
                package1[f'battery_{j+1}'].append(np.concatenate((voltage,current,time),axis=0))
                package2[f'battery_{j+1}'].append(soh)
    CALCE_data[f'package_{i+1}']=package1
    CALCE_SOH[f'package_{i+1}']=package2

In [ ]:
save_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\CALCE_dataset'
#save_data(CALCE_data, os.path.join(save_path, 'CALCE_data.pkl'))
CALCE_data=load_data(os.path.join(save_path, 'CALCE_data.pkl'))
#save_data(CALCE_SOH, os.path.join(save_path, 'CALCE_SOH.pkl'))
CALCE_SOH=load_data(os.path.join(save_path, 'CALCE_SOH.pkl'))

In [ ]:
print(CALCE_data.keys())
print(CALCE_SOH.keys())
print(CALCE_data['package_1'].keys())
print(CALCE_SOH['package_1'].keys())
print(CALCE_SOH['package_1']['battery_1'])


In [ ]:
package='package_1'
battery_soh_plot(CALCE_SOH,CALCE_SOH[package].keys(),package)

In [ ]:
smooth_CALCE_SOH=smooth_soh(CALCE_SOH,method='gaussian',sigma=1)
package='package_1'
battery_soh_plot(smooth_CALCE_SOH,CALCE_SOH[package].keys(),package)
save_data(smooth_CALCE_SOH, os.path.join(save_path, 'CALCE_SOH.pkl'))

In [ ]:
smooth_CALCE_SOH_2=smooth_soh(CALCE_SOH, method='mean')
package='package_1'
battery_soh_plot(smooth_CALCE_SOH_2,CALCE_SOH[package].keys(),package)